# Transformers

A transformer consists of **encoder-decoder** structure, though many models (e.g., BERT, GPT) use only one part.
- The encoder maps an input sequence $x = (x_1, x_2, …, x_n)$, to a sequence of hidden representations.
- The decoder takes these representations and generates an output sequence $y = (y_1, y_2, …, y_m)$, step by step.

Each encoder and decoder consists of multiple stacked layers, each with **self-attention** mechanism. The decoder also includes **cross-attention**, allowing it to attend to encoder outputs.

## Self-Attention Mechanism

Self-attention is at the core of transformers, allowing each token in a sequence to dynamically focus on relevant tokens. The key idea is to compute attention scores that define how much focus each word should give to every other word.

For an input sequence represented as an embedding matrix $X \in \mathbb{R}^{n \times d}$  (where  $n$  is the sequence length and  $d$  is the embedding dimension), self-attention works as follows:

1. Compute three learned projections, **Query**, **Key**, and **Value** matrices for the input tokens:
   $$ Q = XW_Q, \quad K = XW_K, \quad V = XW_V $$

    where:
    - $Q$ (Query): Represents what this token is "searching for" in others.
    - $K$ (Key): Represents what this token "has to offer".
    - $V$ (Value): Represents the actual content of the token.
    - $W_Q, W_K, W_V \in \mathbb{R}^{d \times d_k}$ are learnable weights, projection matrices, where $d_k$ is the dimension of queries and keys.

2. Compute attention scores betweem all tokens using a dot product:

   $$ \text{Scores} = QK^{T} $$

    this results in a matrix $\mathbb{R}^{n \times n}$, where each entry, $S_{ij}$ represents the relevance of token $i$ to token $j$.

3. Scale the cores to prevent large values that can lead to vanishing gradients, we scale the scores by $\sqrt{d_k}$:

    $$ \text{Scaled Scores} = \frac{QK^T}{\sqrt{d_k}} $$

    where $d_k$ is the dimension of queries and keys. The scaling prevents extreme values that could saturate the softmax function.

4. Apply softmax to get attention weights. A softmax function is applied row-wise to normalize the scores into probabilities:

    $$ \text{Attention}(Q, K, V) = \text{softmax} \left( \frac{QK^T}{\sqrt{d_k}} \right) $$


    each row sums to 1, so every token's attention distribution is well-defined probability.

5. Compute weighted sum of values. Each token's final representation is a weighted sum of the values:

   $$\text{Output} = AV$$

    where $A$ (attention matrix) determines how much focus each token places on others.


Self-attention enables transformers to model relationships between tokens in a sequence. It allows each token in a sequence to attend to, or focus on, other tokens, dynamically weighing their importance. Unlike RNNs, which process token sequentially, self-attention enables parallel computation across all tokens at once. Given an input sequence of $n$ tokens, each token computes its attention score with every other token, determining how much "attention" it should give to them.
 
### Multi-Head Attention

Instead of a single attention function, transformers use **multi-head attention**, where $h$ separate attention heads are computed in parallel and concatenated. Each head computes:

$$\text{head}_i = \text{Attention}(XWQ_i, XW_{K_i}, XW_{V_i})$$

The outputs from all heads are concatenated and linearly transformed: 

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, …, \text{head}_h) W_O$$

where $W_O$ is a final learnable projection matrix. Multi-head attention enables learning of different attention patterns, allowing the model to capture diverse relationships between tokens.

## Positional Encoding

Transformers do not have recurrence or convolution, so they need explicit positional encodings to incorporate token order information. One alternative would be to learn a separate positonal embedding vector for each position, just like word embeddings. However this approach does not generalize well to longer sequences than those seen during training. Instead original authors of the paper proposed a **fixed sinusoidal encoding**. For a token at position $p$, its encoding is:

$$\text{PE}_{(p, 2i)} = \sin\left(\frac{p}{10000^{2i/d}}\right)$$
$$\text{PE}_{(p, 2i+1)} = \cos\left(\frac{p}{10000^{2i/d}}\right)$$

where:
- $p$ is the position index (e.g. first token = 0, second token = 1, etc.).
- $i$ indexes the dimension of the embedding.
- $d$ is the total emebedding dimension.

The sinusoids enable:
- Smoothness: Nearby positions have encodings that are close to each other.
- Long-Range Generalization: Since sine and cosine functions are periodic, they help generalize to longer sequences than those during training.
- Unique Encoding: Each position has a unique encoding.

After computing the positional encodings, they are added to the word embeddings:

$$X^\prime = X + \text{PE}$$

this way transformer sees both content (word embedding) and position (encoding). 

## Feed-Forward Network and Normalization

Each transformer layer contains a position-wise feed-forward network (FFN) applied independently to each token:


$$\text{FFN}(x) = \text{ReLU}(x W_1 + b_1) W_2 + b_2$$


where $W_1$ and $W_2$ are learnable weight matrices.

Additionally, *LayerNorm* and *residual connections* are used to stabilize training:

$$\text{LayerNorm}(x + \text{Sublayer}(x))$$

## Transformer Variants

Several transformer models modify the basic architecture:
- BERT (Bidirectional Encoder Representations from Transformers):
    - Uses only the encoder part.
    - Trained with masked language modelling (MLM) and next sentence prediction (NSP).
    - Used for classification, question answering, and sentence embedding.
- GPT (Generative Pre-trained Transformer):
    - Uses only the decoder part.
    - Trained with causal (autoregressive) language modelling.
    - Used for text generation.
- T5 (Text-to-Text Transfer Transformer):
    - Uses a full encoder-decoder.
    - Reformulates all tasks as text-to-text.
- Vision Transformers (ViT):
    - Applies transformers to images by splitting them into patches.
    - Uses self-attention for image classification.

Following is an example implementation of scaled-dot product attention and multi-head attention. Later we'll use these to implement the transformers model.

In [63]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

In [64]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Compute the attention scores and return weighted sum of values.

    Parameters
    ----------
    Q, K, V: Tensorts of shape (batch_size, num_heads, seq_len, d_k)
    mask: Optional mask for padding or casual masking (same shape as attention scores)

    Returns
    -------
    Attention output of shape (batch_size, num_heads, seq_len, d_k)
    """
    d_k = Q.shape[-1]
    scores = torch.matmul(Q, K.transpose(-2, -1)) / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))

    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))

    attention_weights = F.softmax(scores, dim=-1)
    output = torch.matmul(attention_weights, V)

    return output

In [65]:
class MultiHeadAttention(nn.Module):
    """
    Stack multiple attention heads and concatenate their outputs.
    """
    def __init__(self, d_model, num_heads):
        """
        d_model: Embedding dimension.
        num_heads: Number of attention heads.
        """
        super(MultiHeadAttention, self).__init__()

        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # dimension per head

        # linear layers for Q, K, V transformations
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, Q, K, V, mask=None):
        batch_size = Q.shape[0]

        # transform inputs into multiple heads
        def split_heads(x):
            return x.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        Q = split_heads(self.W_q(Q))  # (batch, num_heads, seq_len, d_k)
        K = split_heads(self.W_k(K))
        V = split_heads(self.W_v(V))

        attention_output = scaled_dot_product_attention(Q, K, V, mask)
        attention_output = attention_output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)

        return self.W_o(attention_output)

In [80]:

class PositionalEncoding(nn.Module):
    """
    Use sinusoidal positional encoding as the original paper. 
    """
    def __init__(self, d_model, max_seq_len=5000):
        """
        d_model: Embedding dimension.
        max_len: Maximum sequence length.
        """
        super(PositionalEncoding, self).__init__()

        pe = torch.zeros(max_seq_len, d_model)
        pos = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(pos * div_term) # even indices
        pe[:, 1::2] = torch.cos(pos * div_term) # odd indices

        # register buffer so it's saved with the model 
        # but not updated during training
        self.pe = pe.unsqueeze(0)  # shape: (1, max_seq_len, d_model)
        
    def forward(self, x):
        """
        Add positional encoding to input tensor.
        x: (batch_size, seq_len, d_model)
        """
        return x + self.pe[:, :x.size(1), :]

This is an example usage of multi-head attention and positional encoding layers:

In [81]:
d_model = 64  # embedding dimension
num_heads = 8
seq_len = 10
batch_size = 2

# create random embeddings (word embeddings)
x = torch.rand(batch_size, seq_len, d_model)

pos_encoder = PositionalEncoding(d_model)
x_pos = pos_encoder(x)

attention_layer = MultiHeadAttention(d_model, num_heads)
output = attention_layer(x_pos, x_pos, x_pos)

print(f"input shape: {x.shape}")
print(f"positional encoding shape: {x_pos.shape}")
print(f"self-attention output shape: {output.shape}")

input shape: torch.Size([2, 10, 64])
positional encoding shape: torch.Size([2, 10, 64])
self-attention output shape: torch.Size([2, 10, 64])


Implementing a transformer model:

In [82]:
import torch
import torch.nn as nn

In [83]:
class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        """
        vocab_size: Number of unique tokens in vocabulary
        d_model: Embedding size
        """
        super(TokenEmbedding, self).__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)

    def forward(self, x):
        return self.embedding(x)

Casual self-attention (masked multi-head attention):

We prevent tokens from attending to future tokens by masking the upper-triangle of the attention scores.

In [70]:
def generate_casual_mask(seq_len):
    """
    Creates a lower triangular mask (casual mask) to prevent attending
        to future tokens.
    """
    return torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0)  # (1, 1, seq_len, seq_len)

In [71]:
class CasualSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.multi_head_attention = MultiHeadAttention(d_model, num_heads)

    def forward(self, x):
        seq_len = x.shape[1]
        mask = generate_casual_mask(seq_len).to(x.device)  # (1, 1, seq_len, seq_len)
        return self.multi_head_attention(x, x, x, mask)

A GPT-like model consists of decoder-only blocks with:
- casual self-attention
- feed-forward network
- layer normalization and residula connections

In [72]:
class TransformerDecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        """
        d_model: Embedding size
        num_heads: number of attention heads
        d_ff: feed-forward network size (typically 4 * d_model)
        """
        super().__init__()

        self.attention = CasualSelfAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model)

        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model))
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        attn_out = self.attention(x)
        x = self.norm1(x + attn_out)

        ff_out = self.feed_forward(x)
        x = self.norm2(x + ff_out)

        return x

In [73]:
class GPT(nn.Module):
    """
    Stack multiple decoder blocks and add a final output layer 
        to predict the next token.
    """
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, max_seq_len):
        """
        vocab_size: Size of vocabulary.
        d_model: Embedding size.
        num_heads: Number of attention heads.
        d_ff: Feed-forward network hidden size.
        num_layers: Number of decoder blocks.
        max_seq_len: Maximum sequence length.
        """
        super().__init__()

        self.token_embedding = TokenEmbedding(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_len)

        self.layers = nn.ModuleList([
            TransformerDecoderBlock(d_model, num_heads, d_ff) for _ in range(num_layers)
        ])

        self.lm_head = nn.Linear(d_model, vocab_size)  # final projection to vocabulary

    def forward(self, x):
        x = self.token_embedding(x)
        x = self.positional_encoding(x)

        for layer in self.layers:
            x = layer(x)

        logits = self.lm_head(x)  # (batch, seq_len, vocab_size)
        return logits

Training on Tiny Shakespear dataset:

In [74]:
import requests
import random
import os

In [75]:
if not os.path.exists("shakespear.txt"):
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    response = requests.get(url)
    with open("shakespear.txt", 'w', encoding="utf-8") as f:
        f.write(response.text)

with open("shakespear.txt", 'r', encoding="utf-8") as f:
    text = f.read()

chars = sorted(set(text))
vocab_size = len(chars)
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}

def encode(text):
    return [char_to_idx[ch] for ch in text]

def decode(indices):
    return ''.join([idx_to_char[i] for i in indices])

data = torch.tensor(encode(text), dtype=torch.long)
print(data)
print(chars)

tensor([18, 47, 56,  ..., 45,  8,  0])
['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [76]:
seq_len = 100
batch_size = 32

def get_batch(data, seq_len, batch_size):
    start_indices = torch.randint(0, len(data) - seq_len - 1, (batch_size,))
    x = torch.stack([data[i: i+seq_len] for i in start_indices])
    y = torch.stack([data[i+1: i+seq_len+1] for i in start_indices])
    return x, y

x_batch, y_batch = get_batch(data, seq_len, batch_size)
print(f"batch input shape: {x_batch.shape}")
print(f"batch target shape: {y_batch.shape}")

batch input shape: torch.Size([32, 100])
batch target shape: torch.Size([32, 100])


In [77]:
# hyperparameters
d_model = 128
num_heads = 8
d_ff = 512
num_layers = 6
max_seq_len = 200
num_epochs = 5

# model
device = "cuda" if torch.cuda.is_available() else "cpu"
model = GPT(vocab_size, d_model, num_heads, d_ff, num_layers, max_seq_len).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
loss_fn = nn.CrossEntropyLoss()

# training
for epoch in range(num_epochs):
    x_batch, y_batch = get_batch(data, seq_len, batch_size)
    x_batch, y_batch = x_batch.to(device), y_batch.to(device)

    logits = model(x_batch)  # (batch, seq_len, vocab_size)
    loss = loss_fn(logits.view(-1, vocab_size), y_batch.view(-1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"epoch: {epoch+1}, loss: {loss.item():.4f}")

epoch: 1, loss: 4.3930
epoch: 2, loss: 3.9836
epoch: 3, loss: 3.6857
epoch: 4, loss: 3.5483
epoch: 5, loss: 3.4453


In [78]:
def generate_text(model, start_text, max_new_tokens=200):
    model.eval()
    input_tokens = torch.tensor(encode(start_text), dtype=torch.long).unsqueeze(0).to(device)

    for _ in range(max_new_tokens):
        logits = model(input_tokens)[:, -1, :]
        probs = torch.nn.functional.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        input_tokens = torch.cat([input_tokens, next_token], dim=1)
    
    return decode(input_tokens.squeeze().tolist())

In [79]:
print(generate_text(model, "ROMEO:"))

RuntimeError: The size of tensor a (201) must match the size of tensor b (200) at non-singleton dimension 1

Unlike the original Transformer, which applies LayerNorm after residual connections, GPT 2 applies LayerNorm before them:
- Pre-Normalisation (GPT-2):
    $$x = x + \text{SelfAttention}(\text{LayerNorm}(x))$$
    $$x = x + \text{FeedForward}(\text{LayerNorm}(x))$$

- Post-Normalisation (Standard Transformer):
    $$x = \text{LayerNorm}(x + \text{SelfAttention}(x))$$
    $$x = \text{LayerNorm}(x + \text{FeedForward}(x))$$

this results in more stable training with deep models and better gradient flow in residual connections.

GPT-2 scales residual connections by a factor of $\frac{1}{\sqrt{2}}$, which stabilizes training:

$$x = x + \frac{1}{\sqrt{2}}\text{SelfAttention}(\text{LayerNorm}(x))$$

In [84]:
import torch
import torch.nn as nn
import torch.utils.checkpoint as checkpoint

In [89]:
class GPT2DecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, use_gradient_checkpointing=False):
        super().__init__()
        self.use_gradient_checkpointing = use_gradient_checkpointing

        self.norm1 = nn.LayerNorm(d_model)
        self.attention = CasualSelfAttention(d_model, num_heads)

        self.norm2 = nn.LayerNorm(d_model)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )

    def forward(self, x):
        def _forward(x):
            # apply layernorm before residuals (GPT-2 style)
            x += self.attention(self.norm1(x)) / (2 ** 0.5)
            x += self.feed_forward(self.norm2(x)) / (2 ** 0.5)
            return x

        if self.use_gradient_checkpointing:
            return checkpoint.checkpoint(_forward, x)
        else:
            return _forward(x)

In [98]:
class GPT2(nn.Module):
    def __init__(self, 
                 vocab_size, 
                 d_model, 
                 num_heads, 
                 d_ff, 
                 num_layers, 
                 max_seq_len, 
                 use_gradient_checkpointing=False):
        super().__init__()

        self.token_embedding = TokenEmbedding(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_len)

        self.layers = nn.ModuleList([
            GPT2DecoderBlock(d_model, num_heads, d_ff, use_gradient_checkpointing) for _ in range(num_layers)
        ])

        self.norm_final = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        x = self.token_embedding(x)
        x = self.positional_encoding(x)

        for layer in self.layers:
            x = layer(x)

        x = self.norm_final(x)  # final gpt-2 style norm
        logits = self.lm_head(x)
        return logits

Flash Attention is a highly optimized attention mechanism which reduces memory usage and speeds up training:

In [99]:
class FlashCasualSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_qkv = nn.Linear(d_model, 3 * d_model)

    def forward(self, x):
        batch_size, seq_len, _ = x.shape
        qkv = self.W_qkv(x).view(batch_size, seq_len, self.num_heads, 3, self.d_k).transpose(1, 2)
        return flash_attn_qkvpacked_func(qkv, casual=True)

In [96]:
try:
    from flash_attn import flash_attn_qkvpacked_func
except ImportError:
    FlashCasualSelfAttention = CasualSelfAttention

In [105]:
# mixed-precision training (FP16)
scaler = torch.amp.GradScaler("cuda")
with torch.amp.autocast("cuda"):
    logits = model(x_batch)
    loss = loss_fn(logits.view(-1, vocab_size), y_batch.view(-1))

scaler.scale(loss).backward()
scaler.step(optimizer)
scaler.update()

RuntimeError: one of the variables needed for gradient computation has been modified by an inplace operation: [torch.FloatTensor [32, 128, 256]], which is output 0 of AddBackward0, is at version 24; expected version 23 instead. Hint: enable anomaly detection to find the operation that failed to compute its gradient, with torch.autograd.set_detect_anomaly(True).

In [106]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = GPT2(vocab_size, d_model=256, num_heads=8, d_ff=1024, num_layers=12, max_seq_len=200, use_gradient_checkpointing=False).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, betas=(0.9, 0.95), weight_decay=0.1)
loss_fn = nn.CrossEntropyLoss()

num_epochs = 5
for epoch in range(num_epochs):
    x_batch, y_batch = get_batch(data, seq_len=128, batch_size=32)
    x_batch, y_batch = x_batch.to(device), y_batch.to(device)

    optimizer.zero_grad()
    with torch.cuda.amp.autocast():
        logits = model(x_batch)
        loss = loss_fn(logits.view(-1, vocab_size), y_batch.view(-1))

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    print(f"epoch {epoch+1}, loss: {loss.item():.4f}")

/var/folders/rt/r32hph0n0513b5277cfmjcss3b0c_2/T/ipykernel_66016/3421401308.py:13: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


RuntimeError: one of the variables needed for gradient computation has been modified by an inplace operation: [torch.FloatTensor [32, 128, 256]], which is output 0 of AddBackward0, is at version 24; expected version 23 instead. Hint: enable anomaly detection to find the operation that failed to compute its gradient, with torch.autograd.set_detect_anomaly(True).

In [107]:
def generate_text(model, start_text, max_new_tokens=200, temperature=1.0):
    model.eval()
    input_tokens = torch.tensor(encode(start_text), dtype=torch.long).unsqueeze(0).to(device)

    for _ in range(max_new_tokens):
        with torch.no_grad():
            logits = model(input_tokens)[:, -1, :] / temperature
            probs = torch.nn.functional.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            input_tokens = torch.cat([input_tokens, next_token], dim=1)

    return decode(input_tokens.squeeze().tolist())

print(generate_text(model, "ROMEO:", max_new_tokens=200))

RuntimeError: The size of tensor a (201) must match the size of tensor b (200) at non-singleton dimension 1